# Análise Exploratória de Dados (EDA)

Visualiza a qualidade, distribuição e separabilidade das sequências proteicas
após o pré-processamento pelo módulo 01_ingest.py.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from Bio import SeqIO
from sklearn.manifold import TSNE

sns.set_theme(style='whitegrid', palette='muted')
PROC = '../data/processed'
print('Bibliotecas carregadas.')

## 1. Carregamento dos dados processados

In [ ]:
pos_records = list(SeqIO.parse(os.path.join(PROC, 'positive_clean.fasta'), 'fasta'))
neg_records = list(SeqIO.parse(os.path.join(PROC, 'negative_clean.fasta'), 'fasta'))

with open(os.path.join(PROC, 'data_summary.json')) as f:
    summary = json.load(f)

print(f'Positivos: {len(pos_records)} sequências')
print(f'Negativos: {len(neg_records)} sequências')
print(f'Dados sintéticos usados: {summary["synthetic_data_used"]}')
print(f'Total descartadas: {summary["n_discarded_total"]}')

In [ ]:
# Exemplos de sequências
print('=== Positivos (primeiros 3) ===')
for r in pos_records[:3]:
    print(f'  {r.id}: {str(r.seq)[:60]}...')

print('\n=== Negativos (primeiros 3) ===')
for r in neg_records[:3]:
    print(f'  {r.id}: {str(r.seq)[:60]}...')

## 2. Distribuição de comprimentos

In [ ]:
pos_lengths = [len(str(r.seq)) for r in pos_records]
neg_lengths = [len(str(r.seq)) for r in neg_records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramas sobrepostos
axes[0].hist(pos_lengths, bins=30, alpha=0.6, color='#2c7bb6', label=f'Positivo (n={len(pos_lengths)})')
axes[0].hist(neg_lengths, bins=30, alpha=0.6, color='#d7191c', label=f'Negativo (n={len(neg_lengths)})')
axes[0].set_xlabel('Comprimento (aminoácidos)', fontsize=12)
axes[0].set_ylabel('Frequência', fontsize=12)
axes[0].set_title('Distribuição de Comprimento por Classe', fontsize=13)
axes[0].legend()

# Box plot
axes[1].boxplot([pos_lengths, neg_lengths], labels=['Positivo', 'Negativo'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='#2c7bb6'),
                medianprops=dict(color='#d7191c', linewidth=2))
axes[1].set_ylabel('Comprimento (aminoácidos)', fontsize=12)
axes[1].set_title('Box Plot — Comprimento por Classe', fontsize=13)

plt.tight_layout()
plt.savefig('../reports/eda_length_distribution.png', dpi=150)
plt.show()

print(f'Positivo — mín: {min(pos_lengths)}, máx: {max(pos_lengths)}, média: {np.mean(pos_lengths):.1f}')
print(f'Negativo — mín: {min(neg_lengths)}, máx: {max(neg_lengths)}, média: {np.mean(neg_lengths):.1f}')

## 3. Heatmap — Frequência dos 30 k-mers mais comuns

In [ ]:
import json
from scipy import sparse

X = sparse.load_npz(os.path.join(PROC, 'kmer_matrix.npz'))
y = np.load(os.path.join(PROC, 'labels.npy'))

with open(os.path.join(PROC, 'kmer_vocab.json')) as f:
    vocab = json.load(f)  # {kmer: idx}

idx_to_kmer = {v: k for k, v in vocab.items()}

# Frequência média por classe
pos_mean = np.array(X[y == 1].mean(axis=0)).flatten()
neg_mean = np.array(X[y == 0].mean(axis=0)).flatten()

# Top 30 k-mers por frequência total
total_mean = pos_mean + neg_mean
top30_idx = np.argsort(total_mean)[-30:][::-1]
top30_labels = [idx_to_kmer[i] for i in top30_idx]

heatmap_data = pd.DataFrame(
    {'Positivo': pos_mean[top30_idx], 'Negativo': neg_mean[top30_idx]},
    index=top30_labels
)

fig, ax = plt.subplots(figsize=(6, 12))
sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Frequência relativa média'})
ax.set_title('Top 30 K-mers — Frequência por Classe', fontsize=13)
ax.set_xlabel('Classe', fontsize=11)
ax.set_ylabel('3-mer', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/eda_kmer_heatmap.png', dpi=150)
plt.show()

## 4. t-SNE — Separabilidade das classes no espaço k-mer

In [ ]:
# t-SNE em amostra para viabilidade computacional
MAX_SAMPLE = 500
rng = np.random.default_rng(42)
n = min(MAX_SAMPLE, X.shape[0])
sample_idx = rng.choice(X.shape[0], size=n, replace=False)

X_sample = X[sample_idx].toarray()
y_sample = y[sample_idx]

print(f't-SNE em {n} amostras...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_2d = tsne.fit_transform(X_sample)

fig, ax = plt.subplots(figsize=(8, 6))
colors = {1: '#2c7bb6', 0: '#d7191c'}
labels_map = {1: 'Positivo (marcador)', 0: 'Negativo'}
for cls in [0, 1]:
    mask = y_sample == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=colors[cls], label=labels_map[cls],
               alpha=0.7, s=30, edgecolors='none')

ax.set_title('t-SNE — Espaço K-mer (3-mer, frequência relativa)', fontsize=13)
ax.set_xlabel('Dimensão 1', fontsize=11)
ax.set_ylabel('Dimensão 2', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/eda_tsne.png', dpi=150)
plt.show()

## 5. Conclusões

- **Balanceamento**: verificar se as classes têm amostras comparáveis; caso contrário, considerar pesos de classe nos modelos.
- **Comprimento**: proteínas marcadoras tendem a ter comprimentos distintos de não-marcadoras — isso pode ser um sinal preditivo adicional.
- **Separabilidade no t-SNE**: agrupamentos claros indicam que o espaço k-mer carrega sinal discriminativo suficiente para os classificadores.
- **Frequência k-mer**: diferenças no heatmap entre classes revelam motivos de aminoácidos enriquecidos em marcadores — esses serão refinados pelo SHAP no módulo 05.